
# ISHLR — Unfiltered 695-observation robustness analysis

This notebook checks whether the main iNaturalist conclusion depends on excluding the 56 observations from *Larus smithsonianus*.

It analyzes, with the **same identification program**:

- the initial unfiltered cohort: 695 observations, 22 species, 8 genera;
- the cleaned manuscript cohort: 639 observations, 21 species, 8 genera.

For each cohort it reports:

- the hierarchy-only mean identified width;
- the minimum raw-feasible full-history \(\Gamma\);
- raw feasibility at \(\Gamma\in\{6,8,12,20\}\);
- the full-history minimum relaxation \(\widehat\eta_{n,\mathrm{all}}(\Gamma)\);
- all-history and final-horizon widths under the **same full-history relaxation**;
- tightening relative to hierarchy-only and relative to the final horizon.

The robustness analysis is descriptive. The unfiltered cohort contains a species that failed the trajectory-quality audit, so these results are not intended to replace the cleaned-cohort analysis.


In [ ]:

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linprog

HORIZONS = [0, 1, 7, 30, 90]
FINAL_H = 90
GAMMAS = [6.0, 8.0, 12.0, 20.0]
GAMMA_TOL = 1e-6
MAX_GAMMA = 1e6

OUTDIR = Path("ishf_unfiltered_695_robustness_results")
OUTDIR.mkdir(exist_ok=True)

print("Horizon grid:", HORIZONS)
print("Gamma grid:", GAMMAS)


## 1. Upload the coarse-start validation ZIP

In [ ]:

try:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No ZIP file was uploaded.")
    zip_path = Path(next(iter(uploaded)))
except ImportError:
    candidates = sorted(Path(".").glob("inaturalist_coarse_start_validation_results*.zip"))
    if not candidates:
        raise FileNotFoundError(
            "Place the coarse-start validation ZIP in the working directory."
        )
    zip_path = candidates[0]

print("Using:", zip_path)

with zipfile.ZipFile(zip_path) as zf:
    needed = [
        "reconstructed_coarse_start_trajectories_initial.csv",
        "reconstructed_coarse_start_trajectories.csv",
        "species_quality_audit.csv",
    ]
    missing = [x for x in needed if x not in zf.namelist()]
    if missing:
        raise FileNotFoundError(f"Missing required files in ZIP: {missing}")

    initial = pd.read_csv(
        zf.open("reconstructed_coarse_start_trajectories_initial.csv")
    )
    cleaned = pd.read_csv(
        zf.open("reconstructed_coarse_start_trajectories.csv")
    )
    quality = pd.read_csv(
        zf.open("species_quality_audit.csv")
    )

print(
    "Initial:",
    len(initial), "observations,",
    initial.species_id.nunique(), "species,",
    initial.genus_id.nunique(), "genera"
)
print(
    "Cleaned:",
    len(cleaned), "observations,",
    cleaned.species_id.nunique(), "species,",
    cleaned.genus_id.nunique(), "genera"
)

if (len(initial), initial.species_id.nunique(), initial.genus_id.nunique()) != (695, 22, 8):
    raise RuntimeError("The initial cohort is not the expected 695/22/8 cohort.")

if (len(cleaned), cleaned.species_id.nunique(), cleaned.genus_id.nunique()) != (639, 21, 8):
    raise RuntimeError("The cleaned cohort is not the expected 639/21/8 cohort.")

excluded = quality.loc[quality["exclude"].astype(bool)]
print("\nSpecies excluded by the quality rule:")
print(excluded.to_string(index=False))


## 2. Identification-program helpers

In [ ]:

def build_problem(traj):
    species_df = (
        traj[["species_id","species_name","genus_id","genus_name"]]
        .drop_duplicates()
        .sort_values(["genus_id","species_id"])
        .reset_index(drop=True)
    )

    species_ids = species_df.species_id.astype(int).tolist()
    leaf_nodes = [f"S:{sid}" for sid in species_ids]
    leaf_index = {node:i for i,node in enumerate(leaf_nodes)}
    nL = len(leaf_nodes)

    genera = (
        species_df[["genus_id","genus_name"]]
        .drop_duplicates()
        .sort_values("genus_id")
        .reset_index(drop=True)
    )

    children = {"ROOT":[f"G:{int(g)}" for g in genera.genus_id]}

    for gid, group in species_df.groupby("genus_id"):
        children[f"G:{int(gid)}"] = [
            f"S:{int(sid)}" for sid in group.species_id
        ]

    descendants = {leaf:[leaf] for leaf in leaf_nodes}

    for node, chs in children.items():
        if node != "ROOT":
            descendants[node] = list(chs)

    descendants["ROOT"] = list(leaf_nodes)

    def subtree_vector(node):
        v = np.zeros(nL)
        for leaf in descendants[node]:
            v[leaf_index[leaf]] = 1.0
        return v

    subtree_vec = {
        node:subtree_vector(node)
        for node in descendants
    }

    subtree_states = {}

    for node in descendants:
        if node == "ROOT":
            subtree_states[node] = set(["ROOT"] + children["ROOT"] + leaf_nodes)
        elif node.startswith("G:"):
            subtree_states[node] = set([node] + descendants[node])
        else:
            subtree_states[node] = {node}

    state_cols = {day:f"state_{day}d" for day in HORIZONS}

    crossing = {}

    for parent, chs in children.items():
        crossing[parent] = {}
        for child in chs:
            crossing[parent][child] = {}
            allowed = subtree_states[child]
            for day in HORIZONS:
                crossing[parent][child][day] = float(
                    traj[state_cols[day]].isin(allowed).mean()
                )

    return {
        "traj":traj,
        "species_df":species_df,
        "leaf_nodes":leaf_nodes,
        "leaf_index":leaf_index,
        "nL":nL,
        "children":children,
        "descendants":descendants,
        "subtree_vec":subtree_vec,
        "crossing":crossing,
    }


def fixed_constraints(problem, gamma, horizons, eta=0.0):
    nL = problem["nL"]
    children = problem["children"]
    crossing = problem["crossing"]
    subtree_vec = problem["subtree_vec"]

    A = []
    b = []

    # Final-horizon hierarchy compatibility.
    for parent, chs in children.items():
        for child in chs:
            A.append(-subtree_vec[child])
            b.append(-crossing[parent][child][FINAL_H])

    # Ordered sibling sensitivity inequalities.
    for parent, chs in children.items():
        for day in horizons:
            for child_j in chs:
                for child_k in chs:
                    if child_j == child_k:
                        continue

                    rj = crossing[parent][child_j][day]
                    rk = crossing[parent][child_k][day]

                    A.append(
                        rj * subtree_vec[child_k]
                        - gamma * rk * subtree_vec[child_j]
                    )
                    b.append(
                        eta + (1.0-gamma) * rj * rk
                    )

    return np.asarray(A, dtype=float), np.asarray(b, dtype=float)


def raw_feasible(problem, gamma, horizons):
    A, b = fixed_constraints(problem, gamma, horizons, eta=0.0)
    res = linprog(
        np.zeros(problem["nL"]),
        A_ub=A,
        b_ub=b,
        A_eq=np.ones((1,problem["nL"])),
        b_eq=[1.0],
        bounds=[(0,None)]*problem["nL"],
        method="highs",
    )
    return bool(res.success)


def minimum_relaxation(problem, gamma, horizons):
    nL = problem["nL"]
    children = problem["children"]
    crossing = problem["crossing"]
    subtree_vec = problem["subtree_vec"]

    A = []
    b = []

    # hierarchy constraints: no eta relaxation
    for parent, chs in children.items():
        for child in chs:
            row = np.zeros(nL + 1)
            row[:nL] = -subtree_vec[child]
            A.append(row)
            b.append(-crossing[parent][child][FINAL_H])

    # sensitivity constraints: A p - eta <= baseline
    for parent, chs in children.items():
        for day in horizons:
            for child_j in chs:
                for child_k in chs:
                    if child_j == child_k:
                        continue

                    rj = crossing[parent][child_j][day]
                    rk = crossing[parent][child_k][day]

                    row = np.zeros(nL + 1)
                    row[:nL] = (
                        rj * subtree_vec[child_k]
                        - gamma * rk * subtree_vec[child_j]
                    )
                    row[nL] = -1.0

                    A.append(row)
                    b.append((1.0-gamma) * rj * rk)

    c = np.zeros(nL + 1)
    c[nL] = 1.0

    Aeq = np.zeros((1,nL+1))
    Aeq[0,:nL] = 1.0

    bounds = [(0,None)]*nL + [(0,None)]

    res = linprog(
        c,
        A_ub=np.asarray(A),
        b_ub=np.asarray(b),
        A_eq=Aeq,
        b_eq=[1.0],
        bounds=bounds,
        method="highs",
    )

    if not res.success:
        raise RuntimeError("Minimum-relaxation LP failed.")

    return max(0.0, float(res.x[nL]))


def solve_leaf_bounds(problem, gamma=None, horizons=None, eta=0.0, hierarchy_only=False):
    nL = problem["nL"]

    if hierarchy_only:
        A = []
        b = []
        for parent, chs in problem["children"].items():
            for child in chs:
                A.append(-problem["subtree_vec"][child])
                b.append(-problem["crossing"][parent][child][FINAL_H])
        A = np.asarray(A, dtype=float)
        b = np.asarray(b, dtype=float)
    else:
        A, b = fixed_constraints(problem, gamma, horizons, eta=eta)

    lower = np.empty(nL)
    upper = np.empty(nL)

    for j in range(nL):
        c = np.zeros(nL)
        c[j] = 1.0

        lo = linprog(
            c,
            A_ub=A,
            b_ub=b,
            A_eq=np.ones((1,nL)),
            b_eq=[1.0],
            bounds=[(0,None)]*nL,
            method="highs",
        )
        hi = linprog(
            -c,
            A_ub=A,
            b_ub=b,
            A_eq=np.ones((1,nL)),
            b_eq=[1.0],
            bounds=[(0,None)]*nL,
            method="highs",
        )

        if not lo.success or not hi.success:
            raise RuntimeError(
                f"Bound LP failed for leaf {j}, gamma={gamma}, eta={eta}."
            )

        lower[j] = lo.fun
        upper[j] = -hi.fun

    return lower, upper


def minimum_feasible_gamma(problem, horizons, tol=GAMMA_TOL):
    if raw_feasible(problem, 1.0, horizons):
        return 1.0

    lo = 1.0
    hi = 2.0

    while hi <= MAX_GAMMA and not raw_feasible(problem, hi, horizons):
        lo = hi
        hi *= 2.0

    if hi > MAX_GAMMA:
        return np.inf

    while hi - lo > tol * max(1.0, hi):
        mid = 0.5 * (lo + hi)
        if raw_feasible(problem, mid, horizons):
            hi = mid
        else:
            lo = mid

    return hi


## 3. Analyze both cohorts

In [ ]:

problems = {
    "unfiltered_695": build_problem(initial),
    "cleaned_639": build_problem(cleaned),
}

summary_rows = []
leaf_rows = []

for cohort_name, problem in problems.items():
    hierarchy_bounds = solve_leaf_bounds(problem, hierarchy_only=True)
    hierarchy_width = float(
        (hierarchy_bounds[1] - hierarchy_bounds[0]).mean()
    )

    gamma_min_all = minimum_feasible_gamma(problem, HORIZONS)
    gamma_min_final = minimum_feasible_gamma(problem, [FINAL_H])

    print("\n" + "="*70)
    print(cohort_name)
    print("n =", len(problem["traj"]))
    print("species =", problem["nL"])
    print("hierarchy-only mean width =", hierarchy_width)
    print("minimum raw-feasible Gamma, all horizons =", gamma_min_all)
    print("minimum raw-feasible Gamma, final horizon =", gamma_min_final)

    for gamma in GAMMAS:
        eta_all = minimum_relaxation(problem, gamma, HORIZONS)
        raw_all = raw_feasible(problem, gamma, HORIZONS)
        raw_final = raw_feasible(problem, gamma, [FINAL_H])

        all_bounds = solve_leaf_bounds(
            problem, gamma=gamma, horizons=HORIZONS, eta=eta_all
        )
        final_bounds = solve_leaf_bounds(
            problem, gamma=gamma, horizons=[FINAL_H], eta=eta_all
        )

        all_widths = all_bounds[1] - all_bounds[0]
        final_widths = final_bounds[1] - final_bounds[0]

        all_mean = float(all_widths.mean())
        final_mean = float(final_widths.mean())

        vs_hierarchy = (
            1.0 - all_mean / hierarchy_width
            if hierarchy_width > 0 else np.nan
        )
        vs_final = (
            1.0 - all_mean / final_mean
            if final_mean > 0 else np.nan
        )

        tightened = int(np.sum(all_widths < final_widths - 1e-10))

        summary_rows.append({
            "cohort":cohort_name,
            "n":len(problem["traj"]),
            "species":problem["nL"],
            "gamma":gamma,
            "gamma_min_all":gamma_min_all,
            "gamma_min_final":gamma_min_final,
            "raw_feasible_all":raw_all,
            "raw_feasible_final":raw_final,
            "eta_all":eta_all,
            "hierarchy_mean_width":hierarchy_width,
            "final_mean_width_common_eta":final_mean,
            "all_mean_width_common_eta":all_mean,
            "tightening_vs_hierarchy":vs_hierarchy,
            "tightening_vs_final":vs_final,
            "n_leaf_intervals_tightened":tightened,
        })

        sp = problem["species_df"].copy()
        sp["cohort"] = cohort_name
        sp["gamma"] = gamma
        sp["hierarchy_lower"] = hierarchy_bounds[0]
        sp["hierarchy_upper"] = hierarchy_bounds[1]
        sp["final_lower"] = final_bounds[0]
        sp["final_upper"] = final_bounds[1]
        sp["all_lower"] = all_bounds[0]
        sp["all_upper"] = all_bounds[1]
        sp["final_width"] = final_widths
        sp["all_width"] = all_widths

        leaf_rows.append(sp)

summary_df = pd.DataFrame(summary_rows)
leaf_df = pd.concat(leaf_rows, ignore_index=True)

summary_df.to_csv(
    OUTDIR/"unfiltered_vs_cleaned_gamma_profile.csv",
    index=False
)
leaf_df.to_csv(
    OUTDIR/"unfiltered_vs_cleaned_leaf_bounds.csv",
    index=False
)

display_cols = [
    "cohort","gamma","gamma_min_all","raw_feasible_all","eta_all",
    "hierarchy_mean_width","final_mean_width_common_eta",
    "all_mean_width_common_eta","tightening_vs_hierarchy",
    "tightening_vs_final","n_leaf_intervals_tightened"
]

print("\nMain robustness table:")
print(summary_df[display_cols].to_string(index=False))


## 4. Focused comparison at the principal unrelaxed setting \(\Gamma=12\)

In [ ]:

g12 = summary_df.loc[summary_df["gamma"] == 12.0].copy()

print(g12[[
    "cohort",
    "raw_feasible_all",
    "eta_all",
    "hierarchy_mean_width",
    "final_mean_width_common_eta",
    "all_mean_width_common_eta",
    "tightening_vs_hierarchy",
    "tightening_vs_final",
    "n_leaf_intervals_tightened",
]].to_string(index=False))

u = g12.loc[g12["cohort"]=="unfiltered_695"].iloc[0]
c = g12.loc[g12["cohort"]=="cleaned_639"].iloc[0]

robustness = {
    "unfiltered_gamma12_raw_feasible": bool(u["raw_feasible_all"]),
    "unfiltered_gamma12_eta_all": float(u["eta_all"]),
    "unfiltered_gamma12_tightening_vs_hierarchy": float(u["tightening_vs_hierarchy"]),
    "unfiltered_gamma12_tightening_vs_final": float(u["tightening_vs_final"]),
    "cleaned_gamma12_tightening_vs_hierarchy": float(c["tightening_vs_hierarchy"]),
    "cleaned_gamma12_tightening_vs_final": float(c["tightening_vs_final"]),
}

print("\nFocused Gamma=12 robustness summary:")
print(json.dumps(robustness, indent=2))


## 5. Save a compact manuscript-oriented summary

In [ ]:

def row_for(cohort, gamma):
    return summary_df.loc[
        (summary_df["cohort"] == cohort)
        & (summary_df["gamma"] == float(gamma))
    ].iloc[0]

out_summary = {
    "analysis":"Unfiltered 695-observation robustness analysis",
    "purpose":(
        "Assess whether the qualitative temporal-identification result depends "
        "on excluding Larus smithsonianus."
    ),
    "unfiltered": {
        "n":int(len(initial)),
        "species":int(initial.species_id.nunique()),
        "genera":int(initial.genus_id.nunique()),
        "gamma_min_all":float(
            summary_df.loc[
                summary_df.cohort=="unfiltered_695",
                "gamma_min_all"
            ].iloc[0]
        ),
        "gamma_min_final":float(
            summary_df.loc[
                summary_df.cohort=="unfiltered_695",
                "gamma_min_final"
            ].iloc[0]
        ),
    },
    "cleaned": {
        "n":int(len(cleaned)),
        "species":int(cleaned.species_id.nunique()),
        "genera":int(cleaned.genus_id.nunique()),
        "gamma_min_all":float(
            summary_df.loc[
                summary_df.cohort=="cleaned_639",
                "gamma_min_all"
            ].iloc[0]
        ),
        "gamma_min_final":float(
            summary_df.loc[
                summary_df.cohort=="cleaned_639",
                "gamma_min_final"
            ].iloc[0]
        ),
    },
    "gamma12": robustness,
    "interpretation_note":(
        "The unfiltered cohort includes a species that failed the trajectory-quality "
        "audit. Robustness here means persistence of the qualitative temporal "
        "tightening result, not validation of those anomalous histories."
    ),
}

with open(
    OUTDIR/"unfiltered_695_robustness_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(out_summary, f, indent=2)

archive = "ishf_unfiltered_695_robustness_results.zip"

with zipfile.ZipFile(
    archive,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for p in OUTDIR.iterdir():
        zf.write(p, arcname=p.name)

print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Download the ZIP from the notebook file browser.")
